# 02 · Data cleaning & modelling: brand names, missing months, analysis tables

**Project:** BYD France – Business Intelligence & Market Analytics · **Phase 2 · Notebook 2 of 4**

### Business purpose
Notebook 01 produced a faithful copy of CCFA's tables. It is **not yet analysis-ready**:
* CCFA's brand labels change over time (`ALPHA ROMEO` → `ALFA ROMEO`, `ABARTH` + `FIAT` → `ABARTH/FIAT`, `M.G.`).
* Some brands appear and disappear from the list (e.g. Land Rover was folded into "AUTRES" from March to October 2025).
* Dashboards and SQL need consistent brand names, a date column, and attributes such as *Chinese-owned* for the competitor analysis.

### Inputs → outputs
| Input | Output (`data/processed/`) | Role |
|---|---|---|
| `brand_month_raw.csv` (notebook 01) | `brand_month.csv` | **Fact table**: one row per brand per month, with share of market |
| | `dim_brand.csv` | **Dimension table**: brand → owner group, origin, Chinese-owned flag |
| | `market_month.csv` | **Fact table**: total French new-car market per month |

### Data integrity rules applied here
* Every row has a `value_type`: **actual** (copied from CCFA), **derived** (calculated from actual figures, method stated in `note`), or **not_listed** (brand absent from that month's table → value **missing, never 0**).
* No figure is estimated.

## Step 1 · Set up and load the extracted data

**What it does:** imports pandas and asks you to upload `brand_month_raw.csv` (the output of notebook 01, also in your repo under `data/processed/`).
**What you should see:** `Rows loaded: 1475 | months: 34`.

In [ ]:
import pandas as pd
from google.colab import files

uploaded = files.upload()                 # choose brand_month_raw.csv
raw = pd.read_csv(next(iter(uploaded)))
print("Rows loaded:", len(raw), "| months:", raw.period.nunique())
raw.head()

## Step 2 · Build the brand dimension table

**Simple explanation:** a *dimension table* holds descriptive facts about each brand that don't change month to month: who owns it, where the brand comes from, whether the owner is Chinese. The monthly numbers live in a separate *fact table*. This split (a **star schema**) is how BI tools and SQL databases are normally organised.

**Why these columns:**
* `owner_group` lets us compare *groups* (e.g. Stellantis vs Geely Holding), not just brands.
* `brand_origin` vs `owner_country` keeps nuance: **MG** is a British-origin brand owned by China's SAIC; **Volvo** is Swedish, owned by China's Geely. For "Chinese competition" we use `is_chinese_owned` (owner country = China).
* `is_byd` makes BYD easy to highlight in dashboards.

**Data label:** ownership is general industry knowledge, not from CCFA. Re-check it if a brand changes hands.

In [ ]:
BRANDS = [
    # brand, owner_group, brand_origin, owner_country
    ("Abarth", "Stellantis", "Italy", "Netherlands"), ("Alfa Romeo", "Stellantis", "Italy", "Netherlands"),
    ("Citroën", "Stellantis", "France", "Netherlands"), ("DS", "Stellantis", "France", "Netherlands"),
    ("Fiat", "Stellantis", "Italy", "Netherlands"), ("Jeep", "Stellantis", "USA", "Netherlands"),
    ("Lancia", "Stellantis", "Italy", "Netherlands"), ("Maserati", "Stellantis", "Italy", "Netherlands"),
    ("Opel", "Stellantis", "Germany", "Netherlands"), ("Peugeot", "Stellantis", "France", "Netherlands"),
    ("Renault", "Renault Group", "France", "France"), ("Dacia", "Renault Group", "Romania", "France"),
    ("Alpine", "Renault Group", "France", "France"),
    ("Audi", "Volkswagen Group", "Germany", "Germany"), ("Cupra", "Volkswagen Group", "Spain", "Germany"),
    ("Porsche", "Volkswagen Group", "Germany", "Germany"), ("Seat", "Volkswagen Group", "Spain", "Germany"),
    ("Skoda", "Volkswagen Group", "Czechia", "Germany"), ("Volkswagen", "Volkswagen Group", "Germany", "Germany"),
    ("BMW", "BMW Group", "Germany", "Germany"), ("Mini", "BMW Group", "United Kingdom", "Germany"),
    ("Mercedes-Benz", "Mercedes-Benz Group", "Germany", "Germany"), ("Smart", "Mercedes-Benz / Geely JV", "Germany", "Germany/China"),
    ("Toyota", "Toyota Group", "Japan", "Japan"), ("Lexus", "Toyota Group", "Japan", "Japan"),
    ("Hyundai", "Hyundai Motor Group", "South Korea", "South Korea"), ("Kia", "Hyundai Motor Group", "South Korea", "South Korea"),
    ("Ford", "Ford", "USA", "USA"), ("Nissan", "Nissan", "Japan", "Japan"), ("Honda", "Honda", "Japan", "Japan"),
    ("Mazda", "Mazda", "Japan", "Japan"), ("Suzuki", "Suzuki", "Japan", "Japan"), ("Mitsubishi", "Mitsubishi", "Japan", "Japan"),
    ("Tesla", "Tesla", "USA", "USA"), ("Ferrari", "Ferrari", "Italy", "Italy"),
    ("Jaguar", "Tata Motors (JLR)", "United Kingdom", "India"), ("Land Rover", "Tata Motors (JLR)", "United Kingdom", "India"),
    ("Volvo", "Geely Holding", "Sweden", "China"), ("Lynk & Co", "Geely Holding", "China", "China"),
    ("Zeekr", "Geely Holding", "China", "China"), ("Geely", "Geely Holding", "China", "China"),
    ("BYD", "BYD", "China", "China"), ("MG", "SAIC Motor", "United Kingdom", "China"),
    ("Leapmotor", "Leapmotor (Stellantis-partnered)", "China", "China"), ("Xpeng", "Xpeng", "China", "China"),
    ("Chery", "Chery", "China", "China"), ("Omoda", "Chery", "China", "China"), ("Jaecoo", "Chery", "China", "China"),
    ("Skyworth", "Skyworth", "China", "China"),
    ("Other brands (AUTRES)", "n/a", "n/a", "n/a"),
]

dim_brand = pd.DataFrame(BRANDS, columns=["brand", "owner_group", "brand_origin", "owner_country"])
dim_brand.loc[len(dim_brand)] = ["Fiat + Abarth", "Stellantis", "Italy", "Netherlands"]
dim_brand["is_chinese_owned"] = dim_brand["owner_country"].eq("China")
dim_brand["is_byd"] = dim_brand["brand"].eq("BYD")
print("Brands:", len(dim_brand), "| Chinese-owned:", dim_brand.is_chinese_owned.sum())
dim_brand[dim_brand.is_chinese_owned]

## Step 3 · Map CCFA's labels to one standard name per brand

**What it does:** a lookup dictionary translates every label CCFA has used into a standard brand name.

**Examples of what it fixes:**
| CCFA label | Standard brand | Why |
|---|---|---|
| `ALPHA ROMEO` (Jan–Feb 2024) | Alfa Romeo | CCFA spelling error. YTD continues (800 + 398 = 1 198), so it is the same brand |
| `B.M.W.`, `M.G.`, `MERCEDES` | BMW, MG, Mercedes-Benz | Readable names |
| `ABARTH/FIAT` (Oct 2026) | Fiat + Abarth | CCFA merged two brands (handled in Step 4) |

**Safety check built in:** if a future CCFA file contains a label not in the dictionary, the code **stops with an error** instead of silently dropping the brand.

In [ ]:
LABEL_MAP = {
    "ABARTH": "Abarth", "ALFA ROMEO": "Alfa Romeo", "ALPHA ROMEO": "Alfa Romeo", "CITROEN": "Citroën", "DS": "DS",
    "FIAT": "Fiat", "JEEP": "Jeep", "LANCIA": "Lancia", "MASERATI": "Maserati", "OPEL": "Opel", "PEUGEOT": "Peugeot",
    "RENAULT": "Renault", "DACIA": "Dacia", "ALPINE": "Alpine", "AUDI": "Audi", "CUPRA": "Cupra", "PORSCHE": "Porsche",
    "SEAT": "Seat", "SKODA": "Skoda", "VOLKSWAGEN": "Volkswagen", "B.M.W.": "BMW", "MINI": "Mini",
    "MERCEDES": "Mercedes-Benz", "SMART": "Smart", "TOYOTA": "Toyota", "LEXUS": "Lexus", "HYUNDAI": "Hyundai",
    "KIA": "Kia", "FORD": "Ford", "NISSAN": "Nissan", "HONDA": "Honda", "MAZDA": "Mazda", "SUZUKI": "Suzuki",
    "MITSUBISHI": "Mitsubishi", "TESLA": "Tesla", "FERRARI": "Ferrari", "JAGUAR": "Jaguar", "LAND ROVER": "Land Rover",
    "VOLVO": "Volvo", "LYNK CO": "Lynk & Co", "ZEEKR": "Zeekr", "GEELY": "Geely", "BYD": "BYD", "M.G.": "MG",
    "LEAPMOTOR": "Leapmotor", "XPENG": "Xpeng", "CHERY": "Chery", "OMODA": "Omoda", "JAECOO": "Jaecoo",
    "SKYWORTH": "Skyworth", "AUTRES": "Other brands (AUTRES)",
    "ABARTH/FIAT": "Fiat + Abarth",
}

unknown = sorted(set(raw.loc[~raw.is_total_row, "brand_label"]) - set(LABEL_MAP))
print("Labels in data:", raw.loc[~raw.is_total_row, "brand_label"].nunique(), "| unmapped:", unknown)

## Step 4 · Clean and reshape into analysis tables

The function below does five things. Read the comments in the code alongside this list:

1. **Separates the market total** into its own table (`market_month`).
2. **Jaguar, January 2025:** the PDF leaves the YTD cell blank. In January, YTD equals the month by definition, so we fill it and label it **derived**.
3. **Fiat + Abarth:** CCFA listed them separately until September 2026, then as one line. To keep a continuous trend we build a **combined series for every month** (sum of the two, labelled **derived**). October 2026 is CCFA's own combined line (**actual**). The separate Fiat and Abarth rows are kept for reference but excluded from market totals (`in_market_sum = False`) to avoid double counting.
4. **Complete panel:** creates a row for **every brand × every month**. If a brand wasn't listed that month, the row exists with an **empty value** and `value_type = not_listed`. This makes gaps visible in dashboards instead of hiding them, and stops tools from treating them as zero.
5. **Adds analysis columns:** `date`, `year`, `month`, and **market share** (`share_month_pct`, `share_ytd_pct` = brand ÷ market total from the same CCFA table, so the scope matches → **derived**).

In [ ]:
def clean(raw):
    raw = raw.copy()
    unknown = sorted(set(raw.loc[~raw.is_total_row, "brand_label"]) - set(LABEL_MAP))
    if unknown:
        raise ValueError(f"Unmapped brand labels: {unknown}")

    market = (raw[raw.is_total_row][["period", "registrations_month", "registrations_ytd", "source_id"]]
              .rename(columns={"registrations_month": "market_registrations_month",
                               "registrations_ytd": "market_registrations_ytd"}))

    b = raw[~raw.is_total_row].copy()
    b["brand"] = b["brand_label"].map(LABEL_MAP)
    b["value_type"] = "actual"
    b["note"] = ""

    # Jaguar Jan-2025: YTD blank in PDF; in January YTD = month by definition -> derived
    jan_blank = b["period"].str.endswith("-01") & b["registrations_ytd"].isna()
    b.loc[jan_blank, "registrations_ytd"] = b.loc[jan_blank, "registrations_month"]
    b.loc[jan_blank, "value_type"] = "derived"
    b.loc[jan_blank, "note"] = "YTD blank in source; January YTD = month"

    # Fiat + Abarth: CCFA merged them in Oct-2026. Build a consistent combined series for every month.
    fa = b[b.brand.isin(["Fiat", "Abarth"])].groupby(["period"], as_index=False).agg(
        registrations_month=("registrations_month", "sum"), registrations_ytd=("registrations_ytd", "sum"),
        source_id=("source_id", "first"), source_file=("source_file", "first"))
    fa["brand"] = "Fiat + Abarth"; fa["brand_label"] = "ABARTH + FIAT"; fa["value_type"] = "derived"
    fa["note"] = "sum of separately listed ABARTH and FIAT"
    merged = b[b.brand_label == "ABARTH/FIAT"].copy()
    merged["brand"] = "Fiat + Abarth"; merged["note"] = "reported by CCFA as one combined line"
    b = pd.concat([b[b.brand_label != "ABARTH/FIAT"], fa, merged], ignore_index=True)

    # Complete panel: every brand x every month. Months a brand was not listed stay MISSING (not 0).
    periods = sorted(raw.period.unique())
    brands = sorted(b.brand.unique())
    grid = pd.MultiIndex.from_product([periods, brands], names=["period", "brand"]).to_frame(index=False)
    panel = grid.merge(b, on=["period", "brand"], how="left")
    panel["listed_in_source"] = panel["registrations_month"].notna()
    panel.loc[~panel.listed_in_source, "value_type"] = "not_listed"
    panel.loc[~panel.listed_in_source, "note"] = "not listed separately this month (inside AUTRES or not sold); value unknown, not zero"

    panel = panel.merge(market[["period", "market_registrations_month", "market_registrations_ytd"]], on="period", how="left")
    panel["share_month_pct"] = (panel.registrations_month / panel.market_registrations_month * 100).round(3)
    panel["share_ytd_pct"] = (panel.registrations_ytd / panel.market_registrations_ytd * 100).round(3)

    # Which series to use for totals: exclude Fiat & Abarth separately when the combined series exists (avoid double counting)
    # Fiat + Abarth derived rows are the sum of Fiat and Abarth -> use the combined series, not the components
    panel["in_market_sum"] = panel.listed_in_source & ~panel.brand.isin(["Fiat", "Abarth"])

    panel["year"] = panel.period.str[:4].astype(int)
    panel["month"] = panel.period.str[5:].astype(int)
    panel["date"] = pd.to_datetime(panel.period + "-01")
    cols = ["period", "date", "year", "month", "brand", "brand_label", "registrations_month", "registrations_ytd",
            "market_registrations_month", "market_registrations_ytd", "share_month_pct", "share_ytd_pct",
            "value_type", "listed_in_source", "in_market_sum", "source_id", "note"]
    panel = panel[cols].sort_values(["brand", "period"]).reset_index(drop=True)

    market["year"] = market.period.str[:4].astype(int)
    market["month"] = market.period.str[5:].astype(int)
    market["date"] = pd.to_datetime(market.period + "-01")
    market = market[["period", "date", "year", "month", "market_registrations_month", "market_registrations_ytd", "source_id"]]
    return panel, market

brand_month, market_month = clean(raw)
print("brand_month rows:", len(brand_month), "| brands:", brand_month.brand.nunique())
print(brand_month.value_type.value_counts())

## Step 5 · Validation: do the cleaned brands still add up to the market?

**Rule:** for every month, the sum of brands flagged `in_market_sum` must equal CCFA's market total, for the month and YTD.
**Why:** mapping and merging can accidentally drop or double-count a brand. This proves they didn't.
**How to read it:** `All months reconcile: True` and an empty table below.

In [ ]:
def checks(panel, market):
    s = panel[panel.in_market_sum].groupby("period")[["registrations_month", "registrations_ytd"]].sum()
    m = market.set_index("period")[["market_registrations_month", "market_registrations_ytd"]]
    res = pd.DataFrame({"sum_month": s.registrations_month, "market_month": m.market_registrations_month,
                        "sum_ytd": s.registrations_ytd, "market_ytd": m.market_registrations_ytd})
    res["month_ok"] = res.sum_month == res.market_month
    res["ytd_ok"] = res.sum_ytd == res.market_ytd
    return res

recon = checks(brand_month, market_month)
print("Months:", len(recon), "| All months reconcile:", bool((recon.month_ok & recon.ytd_ok).all()))
recon[~(recon.month_ok & recon.ytd_ok)]

## Step 6 · Spot-checks on the tricky cases

**What it does:** shows the four special cases so you can see the rules working.
**How to read it:**
* **Fiat + Abarth:** September YTD 24 328 + October 2 771 = **27 099**, exactly CCFA's merged October line. The combined series is continuous.
* **Land Rover:** March–October 2025 show **NaN / not_listed**. Correct: it was inside "AUTRES", so its true value is unknown.
* **Jaguar Jan 2025:** YTD = 0, `derived`, with the reason in `note`.
* **BYD:** latest months with market share.

In [ ]:
show = ["period", "brand", "registrations_month", "registrations_ytd", "share_month_pct", "value_type", "note"]
print("Fiat + Abarth"); display(brand_month.query("brand == 'Fiat + Abarth' and period >= '2026-08'")[show])
print("Land Rover");    display(brand_month.query("brand == 'Land Rover' and '2025-02' <= period <= '2025-11'")[show])
print("Jaguar");        display(brand_month.query("brand == 'Jaguar' and period == '2025-01'")[show])
print("BYD");           display(brand_month.query("brand == 'BYD'")[show].tail(6))

## Step 7 · Save the analysis tables

**What it does:** writes the three tables as CSV (dates as `YYYY-MM-DD`, which MySQL, Tableau and Power BI all read correctly) and downloads them.
**Then:** upload them to `data/processed/` on GitHub. These are the tables we load into **MySQL in Phase 4**.

In [ ]:
brand_month.to_csv("brand_month.csv", index=False, date_format="%Y-%m-%d")
market_month.to_csv("market_month.csv", index=False, date_format="%Y-%m-%d")
dim_brand.to_csv("dim_brand.csv", index=False)
print("Saved brand_month:", brand_month.shape, "| market_month:", market_month.shape, "| dim_brand:", dim_brand.shape)
for f in ["brand_month.csv", "market_month.csv", "dim_brand.csv"]:
    files.download(f)

## What this notebook achieved, and what's next

**Done:** a clean **star schema**: one fact table (`brand_month`), one market table (`market_month`) and one brand dimension (`dim_brand`), still reconciling exactly to CCFA's totals.

**Interview talking point:** *"I standardised 52 source labels into 51 brands, built a continuous Fiat + Abarth series across CCFA's mid-year merge, and kept months where a brand wasn't reported as explicit missing values rather than zeros, so market-share trends aren't distorted."*

**Next, notebook 03 (exploratory analysis):** BYD's growth trajectory, seasonality, comparison with Tesla, MG and other Chinese brands, and the first answers to the business questions.